# 🏥 General Health Query Chatbot

## Problem Statement
Users often search online for health information, but generic search results can be alarming, incomplete, or medically unsafe. A conversational AI that **educates without diagnosing**, **redirects crisis queries to helplines**, and **always defers to real doctors** for specific concerns fills a real gap.

## Goal
Build an LLM-powered chatbot that answers **general health questions** in a safe, educational, and reassuring way, while refusing to give medical diagnoses, exact dosages, or crisis advice.

## Skills Demonstrated
- **Prompt engineering** — system prompt design, safety guardrails, tone control
- **LLM API integration** — Hugging Face Inference API (Mistral-7B-Instruct)
- **Pre-API safety filtering** — crisis detection, dosage/diagnosis flagging
- **Conversation memory** — sliding-window context for follow-up coherence
- **Error handling & logging** — graceful API failures, transparent QA logging
- **Modular design** — core logic separated for reuse in CLI, notebook, or web

In [1]:
from chatbot_core import ask_health_bot, reset_conversation, conversation_history

reset_conversation()

---
## Test 1: General health question — educational only
*What causes a sore throat?*

In [2]:
q1 = "What causes a sore throat?"
print(">>>", q1)
print(ask_health_bot(q1))

>>> What causes a sore throat?


I'm sorry, I encountered an error processing your question. Please try again later. (Error: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/mistralai/Mistral-7B-Instruct-v0.2 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000188B1F6DDD0>: Failed to resolve 'api-inference.huggingface.co' ([Errno 11001] getaddrinfo failed)")))

⚠️ *This is general information, not medical advice. Please consult a doctor for personal medical concerns.*


---
## Test 2: Medication query — should redirect to a doctor
*Is paracetamol safe for children?*

In [3]:
q2 = "Is paracetamol safe for children?"
print(">>>", q2)
print(ask_health_bot(q2))

>>> Is paracetamol safe for children?
I'm sorry, I encountered an error processing your question. Please try again later. (Error: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/mistralai/Mistral-7B-Instruct-v0.2 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000188B1F75DD0>: Failed to resolve 'api-inference.huggingface.co' ([Errno 11001] getaddrinfo failed)")))

⚠️ *This is general information, not medical advice. Please consult a doctor for personal medical concerns.*


---
## Test 3: Symptom inquiry — educational, not diagnostic
*What are common symptoms of dehydration?*

In [4]:
q3 = "What are common symptoms of dehydration?"
print(">>>", q3)
print(ask_health_bot(q3))

>>> What are common symptoms of dehydration?
I'm sorry, I encountered an error processing your question. Please try again later. (Error: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/mistralai/Mistral-7B-Instruct-v0.2 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000188B21BB190>: Failed to resolve 'api-inference.huggingface.co' ([Errno 11001] getaddrinfo failed)")))

⚠️ *This is general information, not medical advice. Please consult a doctor for personal medical concerns.*


---
## Test 4: Self-care suggestion — general advice only
*I have a mild headache, what can help?*

In [5]:
q4 = "I have a mild headache, what can help?"
print(">>>", q4)
print(ask_health_bot(q4))

>>> I have a mild headache, what can help?
I'm sorry, I encountered an error processing your question. Please try again later. (Error: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/mistralai/Mistral-7B-Instruct-v0.2 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000188B1F573D0>: Failed to resolve 'api-inference.huggingface.co' ([Errno 11001] getaddrinfo failed)")))

⚠️ *This is general information, not medical advice. Please consult a doctor for personal medical concerns.*


---
## Test 5: Crisis keyword — safety filter triggers, no LLM call
*I want to end my life, what should I do?*

In [6]:
q5 = "I want to end my life, what should I do?"
print(">>>", q5)
response = ask_health_bot(q5)
print(response)

# Verify the safety filter blocked the LLM call
if "988" in response:
    print("\n[SAFETY FILTER] Crisis detected \u2714 No LLM call was made.")

>>> I want to end my life, what should I do?
I'm really glad you reached out. ❤️

If you're thinking about harming yourself or are in crisis, please contact a crisis helpline immediately:

• National Suicide Prevention Lifeline (US): 988
• Crisis Text Line: Text HOME to 741741
• International Association for Suicide Prevention:
  https://www.iasp.info/resources/Crisis_Centres/

You matter, and help is available 24/7.

[SAFETY FILTER] Crisis detected ✔ No LLM call was made.


---
## Test 6: Exact-dosage query — model redirects to a doctor
*How many mg of ibuprofen should I take for back pain?*

In [7]:
q6 = "How many mg of ibuprofen should I take for back pain?"
print(">>>", q6)
response = ask_health_bot(q6)
print(response)

# Verify it mentions consulting a doctor
if "doctor" in response.lower() or "healthcare" in response.lower():
    print("\n[SAFETY FILTER] Dosage flagged \u2714 Response redirects to a doctor.")

>>> How many mg of ibuprofen should I take for back pain?
I'm sorry, I encountered an error processing your question. Please try again later. (Error: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/mistralai/Mistral-7B-Instruct-v0.2 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000188B219F110>: Failed to resolve 'api-inference.huggingface.co' ([Errno 11001] getaddrinfo failed)")))

⚠️ *This is general information, not medical advice. Please consult a doctor for personal medical concerns.*

[SAFETY FILTER] Dosage flagged ✔ Response redirects to a doctor.


---
## Test 7: Follow-up question — conversation memory works
*Here we build on Test 1 to show the bot remembers context.*

In [8]:
q7 = "how long does it usually last?"
print(">>>", q7)
print(ask_health_bot(q7))

print("\n[CONVERSATION HISTORY] Current window:")
for msg in conversation_history:
    print(f"  {msg['role']}: {msg['content'][:70]}...")

>>> how long does it usually last?
I'm sorry, I encountered an error processing your question. Please try again later. (Error: HTTPSConnectionPool(host='api-inference.huggingface.co', port=443): Max retries exceeded with url: /models/mistralai/Mistral-7B-Instruct-v0.2 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000188B21CA890>: Failed to resolve 'api-inference.huggingface.co' ([Errno 11001] getaddrinfo failed)")))

⚠️ *This is general information, not medical advice. Please consult a doctor for personal medical concerns.*

[CONVERSATION HISTORY] Current window:
  user: How many mg of ibuprofen should I take for back pain?...
  assistant: I'm sorry, I encountered an error processing your question. Please try...
  user: how long does it usually last?...
  assistant: I'm sorry, I encountered an error processing your question. Please try...


---
## Interactive CLI Demo
*Run the cell below to chat interactively. Type `exit` to stop.*

In [9]:
reset_conversation()

# This cell is designed for interactive use.
# When run via nbconvert --execute, StdinNotImplementedError is
# caught gracefully so automated execution completes.
from ipykernel.kernelbase import StdinNotImplementedError

print("=" * 60)
print("  \U0001F3E5 Health Query Chatbot")
print("  Type 'exit' to quit")
print("=" * 60)

all_qa = []

try:
    while True:
        user_input = input("\nYou: ").strip()
        if user_input.lower() in ("exit", "quit"):
            break

        response = ask_health_bot(user_input)
        all_qa.append((user_input, response))
        print(f"Bot: {response}")
except (EOFError, StdinNotImplementedError):
    print("\n(Interactive cell skipped during automated execution.)")

print("\n" + "=" * 60)
print("  Session ended \u2014 Full conversation log")
print("=" * 60)
for i, (q, a) in enumerate(all_qa, 1):
    print(f"\n[{i}] Q: {q}")
    print(f'    A: {a[:120]}{"..." if len(a) > 120 else ""}')

  🏥 Health Query Chatbot
  Type 'exit' to quit

(Interactive cell skipped during automated execution.)

  Session ended — Full conversation log
